In [6]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import json

load_dotenv()

# The OpenAI SDK pointed at Groq's URL — this is how you reach the Responses API
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-120b"

In [7]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, model=MODEL, temperature=1.0, stop=None, max_tokens=1000):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        stop=stop,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [8]:
def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt that generate 
    Python, JSON, or Regex specifically for AWS-related tasks. Generate and array of JSON each representing task that requires 
    Python, JSON, or Regex to complete.

    Example output:
    ```json
    [
        {
            'task':'Description of the task'
        },
        ... additional tasks ...
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or Regex pattern.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    response_text = chat(messages, temperature=0.3, stop=["\n```"])

    cleaned = response_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()
    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    return json.loads(cleaned)

In [9]:
dataset = generate_dataset()
dataset

[{'task': "Write a single‑line Python function that takes an AWS S3 object key string and returns the bucket name and object prefix (everything before the last '/' character)."},
 {'task': 'Provide a minimal JSON IAM policy that allows read‑only access (s3:GetObject) to all objects under the bucket named "my‑data‑bucket".'},
 {'task': 'Create a regular‑expression pattern that validates an AWS ARN for a Lambda function (e.g., "arn:aws:lambda:us‑west‑2:123456789012:function:my‑function").'}]

In [10]:
with open("evaluation_dataset.json", "w", encoding="utf-8") as file:
    json.dump(dataset, file, indent=3)

In [31]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop=["\n```"])


    cleaned = eval_text.strip()
    
    if cleaned.startswith("```json"):
        cleaned = cleaned.removeprefix("```json").strip()
    if cleaned.endswith("```"):
        cleaned = cleaned.removesuffix("```").strip()

    return json.loads(cleaned)

In [32]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [33]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [34]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [35]:
with open("evaluation_dataset.json", "r", encoding="utf-8") as file:
    dataset = json.load(file)

results = run_eval(dataset)

results

Average score: 8.333333333333334


[{'output': '**One‑liner**\n\n```python\nparse_s3 = lambda k: (k.partition(\'/\')[0], k.rpartition(\'/\')[0] if \'/\' in k else \'\')\n```\n\n**Explanation**\n\n* `k.partition(\'/\')` splits the string at the **first** “/”.  \n  * `[...,0]` → bucket name (the part before the first slash).\n* `k.rpartition(\'/\')` splits the string at the **last** “/”.  \n  * `[...,0]` → object prefix (everything before the last slash).  \n  * If there is no slash at all we return an empty prefix (`\'\'`).\n\nThe lambda takes a single argument – the S3 object key (e.g. `"my‑bucket/folder/sub/file.txt"` ) – and returns a 2‑tuple:\n\n```python\n>>> parse_s3("my-bucket/folder/sub/file.txt")\n(\'my-bucket\', \'my-bucket/folder/sub\')\n>>> parse_s3("my-bucket")\n(\'my-bucket\', \'\')\n```',
  'test_case': {'task': "Write a single‑line Python function that takes an AWS S3 object key string and returns the bucket name and object prefix (everything before the last '/' character)."},
  'score': 8,
  'reasoning':

In [36]:
print(json.dumps(results, indent=3))

[
   {
      "output": "**One\u2011liner**\n\n```python\nparse_s3 = lambda k: (k.partition('/')[0], k.rpartition('/')[0] if '/' in k else '')\n```\n\n**Explanation**\n\n* `k.partition('/')` splits the string at the **first** \u201c/\u201d.  \n  * `[...,0]` \u2192 bucket name (the part before the first slash).\n* `k.rpartition('/')` splits the string at the **last** \u201c/\u201d.  \n  * `[...,0]` \u2192 object prefix (everything before the last slash).  \n  * If there is no slash at all we return an empty prefix (`''`).\n\nThe lambda takes a single argument \u2013 the S3 object key (e.g. `\"my\u2011bucket/folder/sub/file.txt\"` ) \u2013 and returns a 2\u2011tuple:\n\n```python\n>>> parse_s3(\"my-bucket/folder/sub/file.txt\")\n('my-bucket', 'my-bucket/folder/sub')\n>>> parse_s3(\"my-bucket\")\n('my-bucket', '')\n```",
      "test_case": {
         "task": "Write a single\u2011line Python function that takes an AWS S3 object key string and returns the bucket name and object prefix (every

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)